# RAG 第 14 课：真实版 Hybrid Search

在第 5 课我们用 toy 实现讲过 hybrid search 的思想：
- lexical 抓字面
- dense 抓语义
- 融合两套分数

这一课我们把它升级到真实工程版本：

1. lexical 用 **真正的 BM25**，而不是简单词频
2. dense 用 **本地 embedding 模型**，而不是随机向量
3. 融合用 **RRF（Reciprocal Rank Fusion）**，工业界很常见
4. 在 `squad` 真实数据上对比 lexical-only / dense-only / hybrid
5. 再接上本地 LLM 生成答案，看融合检索对最终答案的影响

重点不是分数高，而是让你看清：**hybrid 到底在什么时候能救场。**

In [ ]:
# 先清理 GPU 缓存。
# 这个 notebook 主要走 LM Studio 的本地服务，本地 GPU 占用很轻。
import gc

try:
    import torch
    if torch.cuda.is_available():
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print('GPU cache cleared.')
    else:
        print('CUDA not available, skip GPU cache clear.')
except Exception as e:
    print(f'Torch not available, skip GPU cache clear. ({e})')

## 1. 连接 LM Studio

继续沿用之前几课的本地 OpenAI 兼容接口。
默认会优先选：
- `qwen3.5-35b-a3b` 作为 chat 模型
- 当前可用的 embedding 模型作为向量模型

In [ ]:
import math
import re
from collections import Counter, defaultdict
from typing import List

import numpy as np
from datasets import load_dataset
from openai import OpenAI

BASE_URL = 'http://10.0.0.63:1234/v1'
API_KEY = 'lm-studio'

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
model_ids = [m.id for m in client.models.list().data]
print('available models:', model_ids)

preferred_chat_models = ['qwen3.5-35b-a3b', 'qwen3.5-27b']
chat_model = next((m for m in preferred_chat_models if m in model_ids), None)
embedding_model = next((m for m in model_ids if 'embed' in m.lower() or 'embedding' in m.lower()), None)

print('chat_model      =', chat_model)
print('embedding_model =', embedding_model)

if chat_model is None:
    raise RuntimeError('没有找到可用的 chat model。')
if embedding_model is None:
    raise RuntimeError('没有找到可用的 embedding model。')

## 2. 加载真实数据集

继续用 `squad` 的 validation 子集。
这个数据集对 hybrid 很友好：很多问题里都包含"实体名 / 数字 / 专有名词"，正好能突出 lexical 信号的价值。

In [ ]:
raw_ds = load_dataset('squad', split='validation[:160]')

context_to_doc_id = {}
documents = []
eval_rows = []

for item in raw_ds:
    context = item['context'].strip()
    if context not in context_to_doc_id:
        doc_id = len(documents)
        context_to_doc_id[context] = doc_id
        documents.append({'doc_id': doc_id, 'text': context, 'title': item['title']})
    else:
        doc_id = context_to_doc_id[context]

    if item['answers']['text']:
        eval_rows.append({
            'question': item['question'].strip(),
            'gold_doc_id': doc_id,
            'reference_answer': item['answers']['text'][0].strip(),
            'title': item['title'],
        })

print('num_documents =', len(documents))
print('num_eval_rows =', len(eval_rows))

## 3. 真实版 BM25

第 5 课的 lexical 分数只是简单词项重叠。这一课我们写一个真正的 BM25：

- `idf` 用对数平滑
- 文档长度归一化（参数 `b`）
- 词频饱和（参数 `k1`）

BM25 在 lexical 检索里几乎是默认基线，理解它会让你以后看任何检索系统都更有底。

In [ ]:
def tokenize(text: str) -> List[str]:
    return re.findall(r'[a-zA-Z0-9]+', text.lower())


class BM25:
    def __init__(self, corpus_tokens: List[List[str]], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus_tokens = corpus_tokens
        self.doc_lens = np.array([len(toks) for toks in corpus_tokens], dtype=np.float32)
        self.avgdl = float(self.doc_lens.mean()) if len(corpus_tokens) > 0 else 0.0
        self.N = len(corpus_tokens)

        # 倒排索引：token -> {doc_id: tf}
        self.postings = defaultdict(dict)
        for doc_id, toks in enumerate(corpus_tokens):
            tf = Counter(toks)
            for token, freq in tf.items():
                self.postings[token][doc_id] = freq

        # idf：包含该 token 的文档数
        self.idf = {}
        for token, posting in self.postings.items():
            df = len(posting)
            # BM25 经典 idf 公式（带 +1 平滑，避免负数）
            self.idf[token] = math.log(1 + (self.N - df + 0.5) / (df + 0.5))

    def score(self, query_tokens: List[str]) -> np.ndarray:
        scores = np.zeros(self.N, dtype=np.float32)
        for token in query_tokens:
            if token not in self.postings:
                continue
            idf = self.idf[token]
            for doc_id, freq in self.postings[token].items():
                dl = self.doc_lens[doc_id]
                denom = freq + self.k1 * (1 - self.b + self.b * dl / self.avgdl)
                scores[doc_id] += idf * (freq * (self.k1 + 1)) / denom
        return scores


doc_tokens = [tokenize(doc['text']) for doc in documents]
bm25 = BM25(doc_tokens)
print('bm25 vocab size =', len(bm25.postings))
print('avg doc length =', round(bm25.avgdl, 2))

## 4. 真实版 dense 检索

embedding 部分和前几课一样：本地 embedding 模型 + 余弦相似度。

In [ ]:
def get_embeddings(texts: List[str], batch_size: int = 16) -> np.ndarray:
    all_vectors = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        response = client.embeddings.create(model=embedding_model, input=batch)
        batch_vectors = [np.array(item.embedding, dtype=np.float32) for item in response.data]
        all_vectors.extend(batch_vectors)
        print(f'embedded {min(start + batch_size, len(texts))}/{len(texts)}')
    return np.vstack(all_vectors)


def l2_normalize(matrix: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    norms = np.clip(norms, 1e-12, None)
    return matrix / norms


doc_texts = [doc['text'] for doc in documents]
doc_embeddings = l2_normalize(get_embeddings(doc_texts, batch_size=16))
print('doc_embeddings shape =', doc_embeddings.shape)

## 5. 三种 retriever

我们把三种检索都封装成统一接口，返回 `(doc_id, score)` 列表。
这样后面对比起来非常直观。

In [ ]:
def lexical_retrieve(question: str, top_k: int = 10):
    scores = bm25.score(tokenize(question))
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx])) for idx in top_indices]


def dense_retrieve(question: str, top_k: int = 10):
    q_emb = l2_normalize(get_embeddings([question], batch_size=1))[0]
    scores = doc_embeddings @ q_emb
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(int(idx), float(scores[idx])) for idx in top_indices]

## 6. 融合：Reciprocal Rank Fusion (RRF)

第 5 课用的是 "分数归一化 + 加权求和"。这种方式的问题在于：
- BM25 和 cosine 的分数尺度完全不一样
- min-max 归一化对 outlier 非常敏感

工业界更常用的是 **RRF**：只看名次，不看绝对分数。

公式很简洁：

$$\text{rrf}(d) = \sum_{r \in retrievers} \frac{1}{k + \text{rank}_r(d)}$$

- `k` 是平滑常数，常用 60
- 名次越靠前，贡献越大
- 同一个 doc 在多套结果里都靠前，融合分就会更高

它的一个很重要的好处是：**完全跨尺度**。

In [ ]:
def rrf_fuse(rankings_list, k: int = 60, top_k: int = 10):
    """rankings_list: 多个 retriever 的 [(doc_id, score), ...] 列表。"""
    fused = defaultdict(float)
    for rankings in rankings_list:
        for rank, (doc_id, _) in enumerate(rankings, start=1):
            fused[doc_id] += 1.0 / (k + rank)
    sorted_items = sorted(fused.items(), key=lambda x: x[1], reverse=True)
    return [(doc_id, score) for doc_id, score in sorted_items[:top_k]]


def hybrid_retrieve(question: str, top_k: int = 10, candidate_k: int = 20):
    lex = lexical_retrieve(question, top_k=candidate_k)
    dense = dense_retrieve(question, top_k=candidate_k)
    return rrf_fuse([lex, dense], k=60, top_k=top_k)

## 7. 先看一条样本：三种检索分别返回了什么

这一格是这节课最直观的部分。
你会清楚看到 lexical / dense / hybrid 在同一条 query 上的差异。

In [ ]:
def show_results(name, results, gold_doc_id):
    print(f'--- {name} ---')
    for rank, (doc_id, score) in enumerate(results[:5], start=1):
        marker = '  <-- GOLD' if doc_id == gold_doc_id else ''
        title = documents[doc_id]['title']
        print(f'rank={rank} | doc_id={doc_id} | score={score:.4f} | title={title}{marker}')


sample = eval_rows[0]
print('question:', sample['question'])
print('gold_doc_id:', sample['gold_doc_id'])
print('reference_answer:', sample['reference_answer'])
print()

lex_top = lexical_retrieve(sample['question'], top_k=10)
dense_top = dense_retrieve(sample['question'], top_k=10)
hybrid_top = hybrid_retrieve(sample['question'], top_k=10)

show_results('lexical (BM25)', lex_top, sample['gold_doc_id'])
print()
show_results('dense (embedding)', dense_top, sample['gold_doc_id'])
print()
show_results('hybrid (RRF)', hybrid_top, sample['gold_doc_id'])

## 8. 接上 LLM 生成答案

为了让对比一直能贯穿到最终输出，我们继续把检索结果交给本地 LLM。
评估时会同时报告 `Hit@1`、`Exact Match`、`Token F1`。

In [ ]:
def build_context_block(doc_ids):
    parts = []
    for i, doc_id in enumerate(doc_ids, start=1):
        doc = documents[doc_id]
        parts.append(f'[Document {i}] title: {doc["title"]}\n{doc["text"]}')
    return '\n\n'.join(parts)


def answer_with_llm(question: str, doc_ids):
    system_prompt = (
        'You are a careful question-answering assistant. '
        'Answer only from the provided context. '
        'If the answer is not supported by the context, reply with NOT_FOUND. '
        'Keep the answer short.'
    )
    user_prompt = (
        f'Question: {question}\n\n'
        f'Context:\n{build_context_block(doc_ids)}\n\n'
        'Return only the answer.'
    )
    response = client.chat.completions.create(
        model=chat_model,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_prompt},
        ],
        temperature=0,
    )
    return response.choices[0].message.content.strip()

## 9. 评估指标

和前几课保持一致：Exact Match + Token F1。
再加上一个非常关键的检索指标：`Hit@1`。

In [ ]:
def normalize_token(token):
    token = token.lower()
    token = re.sub(r'[^a-z0-9]+', '', token)
    return token


def to_tokens(text):
    return [t for t in (normalize_token(x) for x in text.split()) if t]


def normalize_text(text):
    return ' '.join(to_tokens(text))


def exact_match(prediction, reference):
    return 1.0 if normalize_text(prediction) == normalize_text(reference) else 0.0


def token_f1(prediction, reference):
    pred_tokens = to_tokens(prediction)
    ref_tokens = to_tokens(reference)
    pred_counter = Counter(pred_tokens)
    ref_counter = Counter(ref_tokens)
    overlap = sum((pred_counter & ref_counter).values())
    if overlap == 0:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

## 10. 小规模批量评估

因为每条样本要跑 3 次检索 + 3 次生成，这里先跑前 6 条。
你后面想看更稳的数字时再把数量加大。

In [ ]:
eval_subset = eval_rows[:6]
results = []

for i, row in enumerate(eval_subset, start=1):
    print(f'processing {i}/{len(eval_subset)}: {row["question"]}')

    lex_top = lexical_retrieve(row['question'], top_k=10)
    dense_top = dense_retrieve(row['question'], top_k=10)
    hybrid_top = hybrid_retrieve(row['question'], top_k=10)

    lex_ctx = [doc_id for doc_id, _ in lex_top[:3]]
    dense_ctx = [doc_id for doc_id, _ in dense_top[:3]]
    hybrid_ctx = [doc_id for doc_id, _ in hybrid_top[:3]]

    ans_lex = answer_with_llm(row['question'], lex_ctx)
    ans_dense = answer_with_llm(row['question'], dense_ctx)
    ans_hybrid = answer_with_llm(row['question'], hybrid_ctx)

    results.append({
        'question': row['question'],
        'gold_doc_id': row['gold_doc_id'],
        'reference_answer': row['reference_answer'],
        'lex_ids': lex_ctx,
        'dense_ids': dense_ctx,
        'hybrid_ids': hybrid_ctx,
        'lex_hit@1': 1.0 if lex_top[0][0] == row['gold_doc_id'] else 0.0,
        'dense_hit@1': 1.0 if dense_top[0][0] == row['gold_doc_id'] else 0.0,
        'hybrid_hit@1': 1.0 if hybrid_top[0][0] == row['gold_doc_id'] else 0.0,
        'ans_lex': ans_lex,
        'ans_dense': ans_dense,
        'ans_hybrid': ans_hybrid,
        'em_lex': exact_match(ans_lex, row['reference_answer']),
        'em_dense': exact_match(ans_dense, row['reference_answer']),
        'em_hybrid': exact_match(ans_hybrid, row['reference_answer']),
        'f1_lex': token_f1(ans_lex, row['reference_answer']),
        'f1_dense': token_f1(ans_dense, row['reference_answer']),
        'f1_hybrid': token_f1(ans_hybrid, row['reference_answer']),
    })

summary = {
    'Hit@1   ': {
        'lexical': float(np.mean([x['lex_hit@1'] for x in results])),
        'dense  ': float(np.mean([x['dense_hit@1'] for x in results])),
        'hybrid ': float(np.mean([x['hybrid_hit@1'] for x in results])),
    },
    'ExactMatch': {
        'lexical': float(np.mean([x['em_lex'] for x in results])),
        'dense  ': float(np.mean([x['em_dense'] for x in results])),
        'hybrid ': float(np.mean([x['em_hybrid'] for x in results])),
    },
    'TokenF1   ': {
        'lexical': float(np.mean([x['f1_lex'] for x in results])),
        'dense  ': float(np.mean([x['f1_dense'] for x in results])),
        'hybrid ': float(np.mean([x['f1_hybrid'] for x in results])),
    },
}

print('\n=== summary ===')
for metric, sub in summary.items():
    print(metric, sub)

## 11. 看逐条结果

和前面几课一样，逐条对比往往比汇总数字更有信息量。
你会看到三种典型情况：
- lexical 命中、dense 没命中（query 含强字面信号）
- dense 命中、lexical 没命中（query 改写成同义说法）
- hybrid 帮你把两边的胜率都吃下来

In [ ]:
for row in results:
    print('question:', row['question'])
    print('gold_doc_id:', row['gold_doc_id'])
    print('lex_top3   :', row['lex_ids'])
    print('dense_top3 :', row['dense_ids'])
    print('hybrid_top3:', row['hybrid_ids'])
    print(f"hit@1  lex/dense/hybrid: {row['lex_hit@1']}/{row['dense_hit@1']}/{row['hybrid_hit@1']}")
    print('reference_answer:', row['reference_answer'])
    print('ans_lex   :', row['ans_lex'])
    print('ans_dense :', row['ans_dense'])
    print('ans_hybrid:', row['ans_hybrid'])
    print(f"EM  lex/dense/hybrid: {row['em_lex']}/{row['em_dense']}/{row['em_hybrid']}")
    print(f"F1  lex/dense/hybrid: {round(row['f1_lex'],3)}/{round(row['f1_dense'],3)}/{round(row['f1_hybrid'],3)}")
    print('-' * 100)

## 12. 这节课的工程直觉

如果说第 5 课让你"看懂 hybrid 的想法"，这一课让你看懂的是它在真实环境里的几个事实：

1. **BM25 仍然是非常强的基线**，不要因为有了 embedding 就直接扔掉。
2. **dense 不一定永远赢**，对实体名 / 数字 / 缩写的 query，BM25 经常更稳。
3. **RRF 比 "分数加权和" 更稳健**，因为它跳过了分数尺度的问题。
4. hybrid 的真正价值是 **降低最差情况**，而不是把最好情况推得更高。

工程上很常见的一种组合就是：

**Hybrid recall → Rerank → LLM judge** 一条龙。

你前面 12、13 两课其实已经把后面两段练过了。

## 13. 本课小结

这节课你要带走的核心是：

1. 真实 lexical 检索 = BM25，不是简单词频。
2. 真实 dense 检索 = 本地 embedding + 余弦相似度。
3. 工业界更喜欢 RRF 这类"基于名次"的融合方式。
4. hybrid 的卖点是"鲁棒"，不是"刷高分"。

下一课最自然的衔接就是：
**Query Rewriting / Query Expansion**，也就是在检索之前先让 LLM 把 query 改写、扩展，让 hybrid 检索拿到更高质量的输入。